# Oppitunti 11 - Agentista agenttiin (A2A) protokolla


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mikä on A2A-protokolla?

Tämä **Agent-to-Agent (A2A) -protokolla** on avoin standardi, joka mahdollistaa tekoälyagenttien viestinnän,
toistensa löytämisen ja yhteistyön — jopa silloin kun ne on rakennettu eri kehyksiin tai isännöity
eri palveluissa.

Keskeiset käsitteet:

- **Löytäminen** – Agentit julkaisevat *Agent-kortin*, joka kuvaa niiden kyvykkyydet, mikä tekee
  helppoa muiden agenttien (tai orkestroijien) löytää oikea asiantuntija tehtävään.
- **Viestinvälitys** – Agentit vaihtavat jäsenneltyjä viestejä yhteisen protokollan kautta, joten yhden
  agentin pyyntö voidaan ymmärtää ja toteuttaa toisella riippumatta sen sisäisestä
  toteutuksesta.
- **Tehtävän elinkaari** – A2A määrittelee tiloja kuten *lähetetty*, *työn alla*, *valmis*, ja
  *epäonnistunut*, antaen orkestroijalle täydellisen näkyvyyden siitä, miten delegoitu tehtävä etenee.

Tässä oppitunnissa simuloimme A2A-tyylistä yhteistyötä kytkemällä kolme erikoistunutta matkailuagenttia
työnkulkuun, jossa kukin agentti tuo oman asiantuntemuksensa ja välittää tulokset seuraavalle.


## Erikoistuneiden matka-agenttien luominen


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Useiden agenttien yhteistyö työnkulun kautta

Yhdistämme kolme agenttia peräkkäiseen työnkulkuun, joka jäljittelee A2A-viestinvälitystä:

1. **CurrencyExchangeAgent** vastaanottaa käyttäjän pyynnön ja tuottaa valuuttaohjeistuksen.
2. **ActivityPlannerAgent** vastaanottaa rikastetun kontekstin ja lisää aktiviteettisuosituksia.
3. **TravelManagerAgent** yhdistää molemmat syötteet lopulliseksi matkakoosteeksi.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A:n ymmärtäminen tuotantoympäristössä

Tuotantoympäristössä A2A-protokolla mahdollistaa tehokkaita palveluiden välisiä skenaarioita:

| Ominaisuus | Kuvaus |
|---|---|
| **Kehysten välinen yhteentoimivuus** | Yhdellä kehystyksellä rakennettu agentti voi delegoida tehtäviä toisella A2A-yhteensopivalla kehystyksellä rakennetulle agentille, mahdollistaen todellisen organisaatiorajat ylittävän yhteentoimivuuden. |
| **Palvelurajat** | Agentit voivat sijaita erillisissä mikropalveluissa, pilvialueilla tai jopa eri organisaatioissa ja silti tehdä yhteistyötä saumattomasti. |
| **Dynaaminen löydettävyys** | Orkestroija voi kysellä Agent Card -rekisteriä ajon aikana löytääkseen tiettyyn alitehtävään parhaiten soveltuvan asiantuntijan. |
| **Streaming & push notifications** | A2A tukee Server-Sent Events (SSE):ää reaaliaikaisiin edistymisilmoituksiin ja push-ilmoituksiin pitkään kestäville tehtäville. |

Yllä rakentamamme työnkulku on tämän mallin yksinkertaistettu, prosessin sisäinen versio. Todellisessa
käyttöönotossa kukin agentti avaisi HTTP-päätepisteen, julkaisisi Agent Cardin ja kommunikoi
A2A JSON-RPC -protokollan kautta.


## Yhteenveto

Tässä oppitunnissa opit:

1. **Mikä A2A-protokolla on** — avoin standardi agenttien väliseen löytämiseen, viestintään ja tehtävien hallintaan.
2. **Kuinka luoda erikoistuneita agenteja** — valuutanvaihtoagentin, aktiviteettisuunnitteluagentin ja matkahallinnan orkestroijan.
3. **Miten yhdistää agentteja työnkulkuun** — käyttämällä `WorkflowBuilder` mallintamaan peräkkäistä viestinvälitystä agenttien välillä.
4. **Miten A2A toimii tuotannossa** — mahdollistaen eri kehysten ja palveluiden välisen yhteistyön dynaamisella havaitsemisella ja suoratoistopäivityksillä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastuuvapauslauseke:
Tämä asiakirja on käännetty tekoälykäännöspalvelulla Co-op Translator (https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, ota huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä pidetään ensisijaisena lähteenä. Kriittisen tiedon osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai virheellisistä tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
